# Processed E-Commerce Dataset

Download **Orders.csv**, **Customers.csv**, and **Products.csv** from the LMS **Study Material** tab. Run the single code cell below, upload all three files together, and it will download `processed_ecommerce_dataset.csv`.

In [ ]:
# Single-cell Google Colab notebook: Process Orders, Customers, and Products
import pandas as pd
from google.colab import files

# Upload Orders.csv, Customers.csv, and Products.csv together.
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if len(csv_files) < 3:
    raise FileNotFoundError('Upload all three files: Orders.csv, Customers.csv, and Products.csv.')

def choose_file(keyword):
    matches = [name for name in csv_files if keyword in name.lower()]
    if not matches:
        raise FileNotFoundError(f'Could not identify the {keyword.title()} CSV. Ensure its filename includes {keyword!r}.')
    return matches[0]

orders = pd.read_csv(choose_file('order'))
customers = pd.read_csv(choose_file('customer'))
products = pd.read_csv(choose_file('product'))
for data in [orders, customers, products]:
    data.columns = data.columns.str.strip()
pd.set_option('display.max_columns', None)

# Identify fields despite small variations in column naming.
def find_column(data, candidates):
    normalized = {col.lower().replace('_', ' ').replace('-', ' ').strip(): col for col in data.columns}
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    for name, original in normalized.items():
        if any(candidate in name for candidate in candidates):
            return original
    return None

order_customer_key = find_column(orders, ['customer id', 'customerid'])
customer_key = find_column(customers, ['customer id', 'customerid'])
order_product_key = find_column(orders, ['product id', 'productid'])
product_key = find_column(products, ['product id', 'productid'])
date_col = find_column(orders, ['order date', 'date'])
quantity_col = find_column(orders, ['quantity', 'qty'])
price_col = find_column(products, ['price', 'unit price', 'product price'])

if not all([order_customer_key, customer_key, order_product_key, product_key, date_col]):
    raise ValueError('Expected Customer ID, Product ID, and Order Date columns were not found. Check the CSV headers.')

print('=' * 80)
print('PROCESSING E-COMMERCE DATASETS')
print('=' * 80)
print(f'Orders: {orders.shape} | Customers: {customers.shape} | Products: {products.shape}')

# concat() demonstration: split and recombine Orders without changing its records.
midpoint = len(orders) // 2
orders_recombined = pd.concat([orders.iloc[:midpoint], orders.iloc[midpoint:]], ignore_index=True)
print(f'\nconcat() demonstration: {len(orders_recombined):,} order rows recombined from two DataFrames.')

# merge() combines order records with their customer and product information.
processed = orders_recombined.merge(customers, how='left', left_on=order_customer_key, right_on=customer_key, suffixes=('', '_customer'))
processed = processed.merge(products, how='left', left_on=order_product_key, right_on=product_key, suffixes=('', '_product'))

# DateTime operations: convert the order date and derive useful calendar features.
processed[date_col] = pd.to_datetime(processed[date_col], errors='coerce')
processed['Order_Month'] = processed[date_col].dt.month
processed['Order_Month_Name'] = processed[date_col].dt.month_name()
processed['Order_Day'] = processed[date_col].dt.day
processed['Order_Day_of_Week'] = processed[date_col].dt.day_name()

# apply() creates meaningful transformed columns.
if quantity_col:
    processed[quantity_col] = pd.to_numeric(processed[quantity_col], errors='coerce')
    processed['Order_Size'] = processed[quantity_col].apply(lambda q: 'Large' if q >= 5 else ('Medium' if q >= 2 else 'Small'))
if price_col:
    processed[price_col] = pd.to_numeric(processed[price_col].astype(str).str.replace(r'[^0-9.-]', '', regex=True), errors='coerce')
    if quantity_col:
        processed['Total_Sales'] = processed.apply(lambda row: row[quantity_col] * row[price_col], axis=1)

# Clean, readable arrangement: original order data, derived fields, then joined customer/product details.
derived_columns = [col for col in ['Order_Month', 'Order_Month_Name', 'Order_Day', 'Order_Day_of_Week', 'Order_Size', 'Total_Sales'] if col in processed.columns]
base_columns = [col for col in orders.columns if col in processed.columns]
other_columns = [col for col in processed.columns if col not in base_columns + derived_columns]
processed = processed[base_columns + derived_columns + other_columns]

output_file = 'processed_ecommerce_dataset.csv'
processed.to_csv(output_file, index=False)

print('\nFINAL PROCESSED DATASET')
print(f'Shape: {processed.shape}')
display(processed.head())
print('\nMissing values after merging:')
display(processed.isna().sum().to_frame(name='Missing Values').query('`Missing Values` > 0'))
print(f'\nSaved: {output_file}')
files.download(output_file)